# M6.4 · Bellek: aktivasyonlar, checkpointing, gradient accumulation

Eğitimde bellek nereye gidiyor? Parametre, gradyan, Adam durumu ve aktivasyon belleğini elle sayıyor, saf Python'da activation checkpointing'in tepe belleğini ve yeniden hesaplama maliyetini simüle ediyor, NumPy'da checkpoint'li bir geri geçişi kendin yazıp tam geri geçişle aynı gradyanı verdiğini doğruluyor, PyTorch'ta `torch.utils.checkpoint` ile saklanan belleği ölçüyor ve en sonda gradient accumulation'ın büyük batch'le birebir aynı adımı attığını gösteriyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m6-4-memory.html](https://m-gonul.github.io/pytorch-lab/m6-4-memory.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m6_4_memory.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    cc = lambda *a: tuple(need(ns, \"checkpoint_cost\")(*a))\n    R.test(\"16 katman, k = 4\", lambda: ensure(cc([1] * 16, 4) == (7, 12), f\"{cc([1] * 16, 4)}\"))\n    R.test(\"k = 1 (checkpoint yok)\", lambda: ensure(cc([1] * 16, 1) == (16, 0), f\"{cc([1] * 16, 1)}\"))\n    R.test(\"k = n (yalnız son çıktı)\", lambda: ensure(cc([1] * 16, 16) == (16, 15), f\"{cc([1] * 16, 16)}\"))\n    R.test(\"eşit olmayan son segment (n = 10, k = 4)\", lambda: ensure(cc([1] * 10, 4) == (5, 7), f\"{cc([1] * 10, 4)}\"))\n    R.test(\"farklı boyutlar\", lambda: ensure(cc([4, 1, 1, 1, 8, 1], 2) == (11, 3), f\"{cc([4, 1, 1, 1, 8, 1], 2)}\"))\n\n    def bk():\n        rows = []\n        for n in (16, 64, 100):\n            k = need(ns, \"best_k\")([1] * n)\n            p = cc([1] * n, k)[0]\n            rows.append([str(n), str(k), str(p), f\"{2 * math.sqrt(n):.1f}\"])\n            ensure(abs(k - math.sqrt(n)) <= 2, f\"n = {n} için k = {k}; ≈ √n bekleniyordu\")\n        R.table(\"En iyi checkpoint aralığı\", [\"n\", \"k\", \"tepe\", \"2√n\"], rows)\n    R.test(\"best_k ≈ √n\", bk)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"en fazla iki seviye döngü\", lambda: max_loops(src, 2, func=\"grads_ckpt\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    Ws = [rng.normal(size=(6, 6)) / 2.5 for _ in range(10)]\n    x, gout = rng.normal(size=(3, 6)), rng.normal(size=(3, 6))\n\n    def full():\n        G = need(ns, \"grads_full\")(x, Ws, gout)\n        h = 1e-6\n        def L(Ws_):\n            hh = x\n            for W in Ws_:\n                hh = np.tanh(hh @ W)\n            return (hh * gout).sum()\n        for i in (0, 5, 9):\n            num = np.zeros_like(Ws[i])\n            for a in range(6):\n                for b in range(6):\n                    Wp = [w.copy() for w in Ws]; Wp[i][a, b] += h\n                    Wm = [w.copy() for w in Ws]; Wm[i][a, b] -= h\n                    num[a, b] = (L(Wp) - L(Wm)) / (2 * h)\n            close(G[i], num, atol=1e-6, what=f\"W{i}\")\n    R.test(\"grads_full = sayısal türev\", full)\n\n    def ck():\n        G = need(ns, \"grads_full\")(x, Ws, gout)\n        rows = []\n        for k in (1, 2, 3, 4, 10):\n            segs = [list(range(s, min(s + k, 10))) for s in range(0, 10, k)]\n            want = max(j + len(sg) for j, sg in enumerate(segs))\n            Gc, peak = need(ns, \"grads_ckpt\")(x, Ws, gout, k)\n            for a, b in zip(G, Gc):\n                close(a, b, what=f\"k = {k}\")\n            ensure(peak == want, f\"k = {k}: tepe {peak}, beklenen {want}\")\n            rows.append([str(k), str(peak)])\n        R.table(\"10 katman\", [\"k\", \"tepe (saklı çıktı)\"], rows)\n    R.test(\"checkpoint'li gradyanlar aynı, tepe doğru\", ck)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n    blocks = torch.nn.ModuleList(torch.nn.Sequential(torch.nn.Linear(256, 256), torch.nn.Tanh()) for _ in range(16))\n    x = torch.randn(64, 256, requires_grad=True)\n    P = 16 * (256 * 256 + 256) * 4\n\n    def pb():\n        ensure(need(ns, \"param_bytes\")(blocks) == P, f\"{ns['param_bytes'](blocks)} ≠ {P}\")\n    R.test(\"param_bytes\", pb)\n\n    def ob():\n        opt = torch.optim.Adam(blocks.parameters())\n        ensure(need(ns, \"optimizer_state_bytes\")(opt) == 0, \"ilk adımdan önce durum boş\")\n        for p in blocks.parameters():\n            p.grad = torch.zeros_like(p)\n        opt.step()\n        b = ns[\"optimizer_state_bytes\"](opt)\n        n_t = len(list(blocks.parameters()))\n        ensure(2 * P <= b <= 2 * P + 8 * n_t, f\"Adam durumu {b:,} bayt; ≈ 2 × {P:,} bekleniyordu\")\n        return f\"parametre {P:,} bayt, Adam durumu {b:,} bayt (≈ 2×)\"\n    R.test(\"optimizer_state_bytes (Adam ≈ 2× parametre)\", ob)\n\n    def sb():\n        def full():\n            h = x\n            for b in blocks:\n                h = b(h)\n            return h\n        out, full_b = need(ns, \"saved_bytes\")(full)\n        ensure(full_b == 16 * (65536 + 262144 + 65536), f\"checkpoint'siz {full_b:,} bayt\")\n        rows = [[\"yok\", f\"{full_b:,}\"]]\n        ref_grad = torch.autograd.grad(out.sum(), [x] + list(blocks.parameters()))\n        for k in (2, 4, 8):\n            o, b = ns[\"saved_bytes\"](lambda: need(ns, \"run_checkpointed\")(blocks, x, k))\n            ensure(b == (16 // k) * 65536, f\"k = {k}: {b:,} bayt; {(16 // k) * 65536:,} bekleniyordu\")\n            g = torch.autograd.grad(o.sum(), [x] + list(blocks.parameters()))\n            for a, c in zip(g, ref_grad):\n                close(a, c, atol=1e-6, what=f\"k = {k} gradyan\")\n            rows.append([str(k), f\"{b:,}\"])\n        R.table(\"İleri geçişte saklanan bayt (16 blok, B = 64, d = 256)\", [\"checkpoint aralığı\", \"bayt\"], rows,\n                note=\"Checkpoint'siz sayıma ağırlıklar da girer (matmul onları saklar; kopya değil, referans). Checkpoint'li sayım yalnız segment girdileri; geri geçişte segment başına yeniden hesaplanır.\")\n    R.test(\"saved_bytes + run_checkpointed\", sb)\n",
 "e": "import copy\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n    base = torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Linear(64, 5)).double()\n    x, y = torch.randn(70, 20, dtype=torch.float64), torch.randint(0, 5, (70,))\n\n    def eq(opt_cls, **kw):\n        def f():\n            a, b = copy.deepcopy(base), copy.deepcopy(base)\n            oa, ob = opt_cls(a.parameters(), **kw), opt_cls(b.parameters(), **kw)\n            for _ in range(3):\n                need(ns, \"accum_step\")(a, oa, x, y, micro=16)\n                F.cross_entropy(b(x), y).backward(); ob.step(); ob.zero_grad()\n            for p, q in zip(a.parameters(), b.parameters()):\n                close(p, q, atol=1e-10)\n            ensure(all(p.grad is None or p.grad.abs().sum() == 0 for p in a.parameters()), \"adımdan sonra gradyanlar sıfırlanmalı\")\n        return f\n    R.test(\"SGD: 5 mikro-batch (16+16+16+16+6) = tek batch\", eq(torch.optim.SGD, lr=0.1))\n    R.test(\"Adam: 3 adım, aynı parametreler\", eq(torch.optim.Adam, lr=1e-2))\n\n    def bn():\n        torch.manual_seed(0)\n        m = torch.nn.Sequential(torch.nn.Linear(20, 32), torch.nn.BatchNorm1d(32), torch.nn.ReLU(), torch.nn.Linear(32, 5)).double()\n        a, b = copy.deepcopy(m), copy.deepcopy(m)\n        oa, ob = torch.optim.SGD(a.parameters(), lr=0.1), torch.optim.SGD(b.parameters(), lr=0.1)\n        need(ns, \"accum_step\")(a, oa, x, y, micro=16)\n        F.cross_entropy(b(x), y).backward(); ob.step()\n        diff = max((p - q).abs().max().item() for p, q in zip(a.parameters(), b.parameters()))\n        ensure(diff > 1e-6, \"BatchNorm'lu modelde fark bekleniyordu\")\n        return f\"BatchNorm'lu modelde en büyük parametre farkı {diff:.2e}: istatistikler mikro-batch üzerinden\"\n    R.test(\"BatchNorm istisnası\", bn)\n\n    def mem():\n        m = torch.nn.Sequential(torch.nn.Linear(256, 1024), torch.nn.GELU(), torch.nn.Linear(1024, 10))\n        opt = torch.optim.SGD(m.parameters(), lr=0.0)\n        xb, yb = torch.randn(512, 256), torch.randint(0, 10, (512,))\n        rows = []\n        for micro in (512, 128, 32):\n            peak = [0]\n            cur = [0]\n\n            def pack(t):\n                cur[0] += t.numel() * t.element_size(); peak[0] = max(peak[0], cur[0]); return t\n\n            def unpack(t):\n                return t\n            orig_backward = torch.Tensor.backward\n\n            def bw(self, *a, **k):\n                cur[0] = 0\n                return orig_backward(self, *a, **k)\n            torch.Tensor.backward = bw\n            try:\n                with torch.autograd.graph.saved_tensors_hooks(pack, unpack):\n                    need(ns, \"accum_step\")(m, opt, xb, yb, micro)\n            finally:\n                torch.Tensor.backward = orig_backward\n            rows.append([str(micro), f\"{512 // micro}\", f\"{peak[0]:,}\"])\n        ensure(int(rows[-1][2].replace(\",\", \"\")) < int(rows[0][2].replace(\",\", \"\")), \"küçük mikro-batch daha az tepe bellek\")\n        R.table(\"Mikro-batch başına tepe saklanan bayt (B = 512)\", [\"mikro-batch\", \"geçiş\", \"tepe bayt\"], rows,\n                note=\"Ağırlıklar her mikro-batch'te saklananlar arasında sayılır (sabit kısım); aktivasyon kısmı mikro-batch ile doğrusal küçülür.\")\n    R.test(\"tepe aktivasyon mikro-batch ile düşüyor\", mem)\n"
}

In [ ]:
import importlib, labcheck, checks_m6_4_memory as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Eğitimde GPU belleği dört kaleme gider:

| kalem | fp32 Adam, parametre başı |
|---|---|
| ağırlıklar | 4 bayt |
| gradyanlar | 4 bayt |
| Adam $m$, $v$ | 8 bayt |
| aktivasyonlar | batch, dizi uzunluğu ve derinlikle **ölçeklenir** |

Aktivasyonlar, geri geçiş için saklanan ara tensörler (M6.1). Büyük batch'lerde ve uzun dizilerde baskın kalem genelde budur.

**activation checkpointing:** ileri geçişte yalnız bazı katmanların çıktısını sakla; geri geçişte bir segmente gelince, onun ara çıktılarını segment başından **yeniden hesapla**. $n$ katmanlı zincirde her $k$ katmanda bir saklarsan tepe bellek $\approx n/k+k$, ek hesap $\approx$ bir ileri geçiş.

**Gradient accumulation:** bellek tek seferde 64 örneğe yetmiyorsa, 4 mikro-batch × 16 ile gradyanları biriktir ve bir kez adım at. Ortalama kayıp için her mikro-batch kaybını mikro-batch sayısına böl.

## 1 · Elle

Son segment işlenirken: 4 segment sonu hâlâ saklı (önceki segmentlerin geri geçişi için gerekli), artı son segmentin yeniden hesaplanan 3 ara çıktısı. Sonraki segmentlere geçtikçe artık gerekmeyen segment sonları serbest bırakılır, bu yüzden tepe ilk işlenen segmentte.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**GPT-2 small, 124,439,808 parametre, fp32 Adam: ağırlık + gradyan + iki moment**

- `adam`: Toplam bayt (aktivasyonsuz)

**16 katmanlı zincir, her katman çıktısı 1 birim. Her $k=4$ katmanda bir çıktıyı sakla (segment sonu)**

- `nock`: Checkpoint'siz: geri geçiş başında saklı çıktı sayısı
- `peak`: Checkpoint'li tepe: son segment işlenirken (4 segment sonu + 3 yeniden hesaplanan)
- `rec`: Toplam yeniden hesaplanan katman
- `bytes`: B = 64, genişlik 256, fp32: bir katman çıktısı kaç bayt?

In [ ]:
cevaplar = dict(
    adam=None,
    nock=None,
    peak=None,
    rec=None,
    bytes=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"adam":[1991036928,0],"nock":[16,0],"peak":[7,0],"rec":[12,0],"bytes":[65536,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python simülasyonu. Katmanlar $0..n-1$, katman $i$'nin çıktı boyutu `sizes[i]`. Her $k$ katmanda bir **segment**; segmentin son katmanının çıktısı saklanır (segment sonu).

- `checkpoint_cost(sizes, k)` → `(tepe, yeniden_hesap)`:
  - Geri geçiş segmentleri **sondan başa** işler. Segment $j$ işlenirken bellekte: segment $0..j$'nin sonları + segment $j$'nin yeniden hesaplanan diğer çıktıları.
  - `tepe` = bunun $j$ üzerinden maksimumu; `yeniden_hesap` = tüm segmentlerde (uzunluk − 1) toplamı.
  - $k=1$ checkpoint'siz durumdur: tepe = `sum(sizes)`, yeniden hesap 0.
- `best_k(sizes)` → tepe belleği en küçük $k$ (eşitlikte küçük $k$).

In [ ]:
%%writefile stage_b.py
def checkpoint_cost(sizes, k):
    ...


def best_k(sizes):
    ...


print(checkpoint_cost([1] * 16, 4))   # (7, 12)
print(checkpoint_cost([1] * 16, 1))   # (16, 0)
print(best_k([1] * 16))               # 3 (k = 3 ve 4 tepe 7 ile berabere; 3 daha az yeniden hesap)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy; tanh zinciri $h_{i+1}=\tanh(h_iW_i)$, çıkışa gelen gradyan `gout`.

- `grads_full(x, Ws, gout)` — tüm $h_i$'leri sakla, geri geçiş: $\ \delta=g\odot(1-h_{i+1}^2)$, $\ \nabla W_i=h_i^\top\delta$, $\ g\leftarrow\delta W_i^\top$.
- `grads_ckpt(x, Ws, gout, k)` → `(grads, peak)`. İleri geçişte yalnız segment sonlarını (ve `x`'i) sakla; geri geçişte her segmenti başından yeniden hesapla. `peak` = aynı anda saklı **katman çıktısı** sayısının maksimumu (`x` hariç), b'deki tanımla aynı.

Katmanlar/segmentler üzerinde döngü serbest (en fazla iki seviye).

In [ ]:
%%writefile stage_c.py
import numpy as np


def grads_full(x, Ws, gout):
    ...


def grads_ckpt(x, Ws, gout, k):
    ...


rng = np.random.default_rng(0)
Ws = [rng.normal(size=(8, 8)) / 3 for _ in range(16)]
x, gout = rng.normal(size=(4, 8)), rng.normal(size=(4, 8))
g1 = grads_full(x, Ws, gout)
g2, peak = grads_ckpt(x, Ws, gout, 4)
print(max(np.abs(a - b).max() for a, b in zip(g1, g2)), peak)   # ~0, 7

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `param_bytes(model)` — parametrelerin bayt toplamı.
- `optimizer_state_bytes(opt)` — `opt.state` içindeki tüm tensörlerin bayt toplamı (Adam'da `exp_avg`, `exp_avg_sq` ve `step`).
- `saved_bytes(fn)` → `(çıktı, bayt)`: `fn()` sırasında saklanan tensörlerin baytı (`saved_tensors_hooks`).
- `run_checkpointed(blocks, x, k)` — `blocks` (`nn.ModuleList`) her $k$ blokta bir segment, her segment `torch.utils.checkpoint.checkpoint(…, use_reentrant=False)` ile.

In [ ]:
%%writefile stage_d.py
import torch
from torch.utils.checkpoint import checkpoint


def param_bytes(model):
    ...


def optimizer_state_bytes(opt):
    ...


def saved_bytes(fn):
    ...


def run_checkpointed(blocks, x, k):
    ...


torch.manual_seed(0)
blocks = torch.nn.ModuleList(torch.nn.Sequential(torch.nn.Linear(256, 256), torch.nn.Tanh()) for _ in range(16))
x = torch.randn(64, 256, requires_grad=True)
print(param_bytes(blocks))
print(saved_bytes(lambda: run_checkpointed(blocks, x, 4))[1])   # 4 segment girdisi × 65536

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`accum_step(model, opt, x, y, micro)` — `x`'i `micro` boyutlu parçalara böl; her parça için `loss = F.cross_entropy(…) * len(parça) / len(x)` ile `backward` (gradyanlar birikir), en sonda tek `opt.step()` ve `opt.zero_grad()`. Son parça daha küçük olabilir: ağırlık `len(parça) / len(x)` bu yüzden.

Kontrol, parametrelerin tek büyük batch adımıyla birebir aynı olduğunu ve mikro-batch küçüldükçe tepe aktivasyonun düştüğünü gösterir.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def accum_step(model, opt, x, y, micro):
    ...


torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(20, 64), torch.nn.ReLU(), torch.nn.Linear(64, 5))
opt = torch.optim.SGD(model.parameters(), lr=0.1)
x, y = torch.randn(70, 20), torch.randint(0, 5, (70,))
accum_step(model, opt, x, y, micro=16)
print(F.cross_entropy(model(x), y).item())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Chen ve ark. (2016): alt-doğrusal bellekle derin ağ eğitimi

[Chen, Xu, Zhang, Guestrin — Training Deep Nets with Sublinear Memory Cost (2016)](https://arxiv.org/abs/1604.06174)

√n kontrol noktası stratejisini (Bölüm 4) n katmanlı bir MLP ya da Transformer yığınında torch.utils.checkpoint ile uygula: segment boyutu k için tepe bellek O(n/k + k) ve ek hesap; k'ya göre ölçülen bellek–süre eğrisini çiz ve en iyi k ≈ √n'yi doğrula. Makalenin bellek planlama (in-place, bellek paylaşımı) fikirlerinden birini torch.cuda.memory_stats / tracemalloc ile ölçerek göster.


**Adımlar**

1. Tepe belleği torch.cuda.max_memory_allocated ya da CPU'da özel ayırıcı sayacıyla ölç.

**Kabul ölçütleri**

- Bellek–k grafiği kuramla uyumlu.
- √n seçiminde ek hesap ≈ bir ileri geçiş.

## Kaynaklar

- [Chen ve ark. — Training Deep Nets with Sublinear Memory Cost](https://arxiv.org/abs/1604.06174) — arXiv
- [PyTorch Docs — torch.utils.checkpoint](https://docs.pytorch.org/docs/stable/checkpoint.html) — BSD-3-Clause
- [Rajbhandari ve ark. — ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054) — arXiv
- [Stanford CS336 — Lecture: Resource accounting](https://stanford-cs336.github.io/) — doğrulanmadı
- [Hugging Face — The Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook) — doğrulanmadı